# 01 · Build — SOC Alert-Triage Assistant

**Vertical:** Cybersecurity · **Use case:** alert-triage · **ADLC stage:** Build

A SOC analyst needs a **grounded triage report**: severity, disposition, escalation decision,
rationale, containment, uncertainty, and citations. We build that agent with `create_agent`: a ReAct
loop over synthetic alert evidence, asset context, incident history, and SOC guidance.

![Architecture](images/01_build.png)

> Self-directed module · ~10 min. By the end you can:
> - build a cited SOC triage agent with `create_agent`
> - wire four grounding tools for alert evidence, assets, incident history, and playbooks
> - enforce a typed report with `response_format=` instead of parsing free text
> - render the compiled graph and open traces / prompt in LangSmith
>
> 🧭 **First lab in the track — runs standalone.** All fixtures are **synthetic / fictional**: no real
> incidents, customers, IPs, PII, secrets, or containment actions.

| Tool | Does |
|---|---|
| `get_alert` | alert evidence by alert ID |
| `get_asset_context` | business criticality, owner, exposure, classification |
| `search_incident_history` | prior synthetic incident matches |
| `search_soc_kb` | severity rubric, playbook, false-positive guidance, containment policy |

## 0. Setup

> 💡 `override=True` lets this lab's `.env` win over globally-set credentials. The printed
> **workspace** is where traces and Prompt Hub links route.

In [ ]:
import json
import os
import sys
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(), override=True)  # FIRST — before any model client is constructed
if "gateway.smith.langchain.com" in os.environ.get("OPENAI_BASE_URL", ""):
    os.environ.pop("OPENAI_BASE_URL", None)  # use the provider key directly, not an injected gateway

_root = Path.cwd()
while _root != _root.parent and not (_root / "utils").is_dir():
    _root = _root.parent
sys.path.insert(0, str(_root))  # import utils/ and alert_triage_graph.py from the content root
DATA = _root / "data"

print("workspace:", os.getenv("LANGSMITH_WORKSPACE_ID"), "| tracing:", os.getenv("LANGSMITH_TRACING"))

## 1. Inspect the synthetic SOC fixtures

The agent can only ground itself in committed fixtures: alert evidence, asset context, prior incidents,
and Markdown SOC guidance.

In [ ]:
alerts = json.loads((DATA / "alerts.json").read_text())
assets = json.loads((DATA / "assets.json").read_text())
incidents = json.loads((DATA / "incidents.json").read_text())
kb_docs = sorted((DATA / "kb").glob("*.md"))

print("alerts  :", len([k for k in alerts if not k.startswith("_")]))
print("assets  :", len([k for k in assets if not k.startswith("_")]))
print("incidents:", len([k for k in incidents if not k.startswith("_")]))
print("kb docs :", [p.name for p in kb_docs])

In [ ]:
alert = alerts["ALT-1001"]
print(alert["title"])
print("assets:", alert["asset_ids"], "| prior incidents:", alert["related_incident_ids"])
print("evidence:")
for item in alert["evidence"]:
    print(" -", item)

## 2. Inspect the grounding tools

These four tools are pure-Python lookups/searches over the same fixtures — no SQL, shell, network, or
real containment. Try each one before giving it to the agent.

In [ ]:
from alert_triage_graph import alert_triage_tools, get_alert, get_asset_context, search_incident_history, search_soc_kb

tools = alert_triage_tools()
print("tools:", [t.name for t in tools])

In [ ]:
print(get_alert.invoke({"alert_id": "ALT-1001"})[:700])

In [ ]:
# ↑ that was a *tool call*. Creating the tool is just as simple — `get_alert` (imported above) is defined
# exactly like this in alert_triage_graph.py: the @tool decorator makes a plain function callable by the
# agent, and its docstring becomes the description the model reads to decide when to use it.
# from langchain_core.tools import tool
#
# @tool
# def get_alert_demo(alert_id: str) -> str:
#     """Get one alert by ID, e.g. `ALT-1001`."""  # ← this docstring is the tool's model-facing description
#     return str(alerts.get(alert_id.strip().upper(), "no such alert"))

# print(get_alert_demo.invoke({"alert_id": "ALT-1001"}))

In [ ]:
# print(get_asset_context.invoke({"query": "AST-IDP-01, AST-FIN-LAP-07"})[:900])

In [ ]:
# print(search_incident_history.invoke({"query": "OAuth Mail.Read MFA fatigue finance"})[:900])

In [ ]:
# print(search_soc_kb.invoke({"query": "OAuth abuse privileged identity escalation containment"})[:900])

## 3. Load the SOC analyst prompt

The prompt sets the severity/disposition taxonomy, citation rules, uncertainty policy, and required
YAML-like report format.

In [ ]:
from alert_triage_graph import PROMPT_NAME, PROMPT_PATH
from utils.prompts import get_prompt, prompt_url

system_prompt = get_prompt(PROMPT_NAME, fallback=PROMPT_PATH.read_text().strip())
print(system_prompt)

## 4. Build the agent

The report's shape is a **Pydantic schema**, not a template in the prompt. Passed as `response_format=`,
LangChain enforces the fields + allowed values and returns a parsed object — so the prompt stays role +
policy (notice §3 has no output template). **The schema is code.**

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class AlertTriageReport(BaseModel):
    severity: Literal["Critical", "High", "Medium", "Low", "Informational"]
    disposition: Literal[
        "confirmed_incident", "likely_incident", "needs_review", "benign_false_positive", "blocked_attempt"
    ]
    escalate: bool = Field(description="True when the rubric/containment requires human escalation or approval.")
    rationale: list[str] = Field(description="Grounded bullets citing alert evidence, asset context, or incident/KB guidance.")
    recommended_containment: list[str] = Field(description="Specific, scoped, reversible actions; empty if none warranted.")
    uncertainty: str = Field(description="What remains unknown and the next non-destructive check.")
    citations: list[str] = Field(description="Fixture IDs / KB filenames used: alert ID, asset/incident IDs, kb *.md.")

One clean keyword-arg call: model + tools + prompt + the response schema in, a compiled ReAct agent out.

In [ ]:
from langchain.agents import create_agent

from utils.models import get_model

model = get_model()  # default model comes from utils/models.py; swap it once there
# Same choice, set inline instead of via the helper (uncomment + `from langchain.chat_models import init_chat_model`):
# model = init_chat_model("openai:gpt-5.4-mini")  # ← this is what get_model() resolves to by default

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt,
    response_format=AlertTriageReport,  # ← enforce the typed report; no format instructions in the prompt
)

**The agent's shape** — `create_agent` returns a compiled LangGraph graph; render it inline.

> 💡 Adding `response_format` gives the `model` node two extra conditional edges (the stray dashed
> arrows): a **`model → model`** self-loop and **`tools → __end__`**. That's the structured-output path —
> after reasoning + any tool calls, the agent re-enters `model` to produce/validate the typed
> `AlertTriageReport` before ending. It's not a bug; without `response_format` the graph is just the plain
> `model ↔ tools → __end__` ReAct loop.

In [ ]:
from IPython.display import Image, display

display(Image(agent.get_graph().draw_mermaid_png()))

## 5. Run one alert triage

`ask()` invokes the agent and prints a clickable **LangSmith trace** link when tracing is enabled.
Because the agent has a `response_format`, it returns a validated `AlertTriageReport` — LangChain puts
it in `result["structured_response"]`, which `ask()` surfaces here.

In [ ]:
from utils.trace import ask

report = ask(agent, "Triage ALT-1001. Provide severity, disposition, escalation decision, rationale, and containment.")

It's a typed `AlertTriageReport`, not free text — so we lay the fields out as a readable analyst handoff
(access any field directly, e.g. `report.severity`; no parsing).

In [ ]:
print(f"severity    : {report.severity}")
print(f"disposition : {report.disposition}")
print(f"escalate    : {report.escalate}")
print("\nrationale:")
for item in report.rationale:
    print(f"  • {item}")
print("\ncontainment:")
for item in report.recommended_containment:
    print(f"  • {item}")
print(f"\nuncertainty : {report.uncertainty}")
print(f"citations   : {', '.join(report.citations)}")

## 6. Check a false-positive case

Good triage avoids over-containment: blocked sanctioned scanner activity should not become a major
incident just because a detector fired.

In [ ]:
fp = ask(agent, "Triage ALT-1005. Is this a real VPN incident or a false positive? Keep the report concise.")
print(f"{fp.severity} / {fp.disposition} · escalate={fp.escalate}")
print("uncertainty:", fp.uncertainty)

## 7. LangSmith links

The runs above printed **trace** links. The prompt link opens the Hub entry for edit / Playground,
scoped to this workspace.

In [ ]:
print("📝 prompt:", prompt_url(PROMPT_NAME))

## 8. Bonus — try the prompt in the Playground

Open the prompt link above, then click **▶ Playground** (top-right, next to **Fork**) to load this exact
prompt against a model — you can edit the wording and re-run it live, no code changes.

![Open the Playground from Prompt Hub](images/01-langsmith-ui_playground.png)

Add a **Human** message with the same input we ran in §5 and hit **Start**:

```text
Triage ALT-1001. Provide severity, disposition, escalation decision, rationale, and containment.
```

> 💡 The Playground runs the **prompt alone** — no tools, no `response_format`. So the model can't call
> `get_alert` to fetch `ALT-1001`, and the reply isn't schema-typed. That grounding and the typed report
> come from the **agent** (tools + `response_format`) you built above. Use the Playground to iterate on
> wording, then pull the edit back into the lab with `get_prompt()`.
>
> 🧪 To try the prompt **with the tools** — the full grounded, typed agent — run the `alert_triage_agent`
> graph in **LangGraph Studio** (`uv run langgraph dev`, then open Studio). Lab 03 walks through it.

## Recap

| Topic | API |
|---|---|
| Build a ReAct agent | `create_agent(model=, tools=, system_prompt=)` |
| Grounding tools | `get_alert`, `get_asset_context`, `search_incident_history`, `search_soc_kb` |
| Typed output (schema is code) | `create_agent(…, response_format=AlertTriageReport)` → `result["structured_response"]` |
| Render the graph | `agent.get_graph().draw_mermaid_png()` |
| Traced runs + prompt link | `ask()` · `prompt_url()` |

**Next:** `02_evaluate.ipynb` — score triage reports against the reference alert dataset.